# Perfiles de viento y alcance geográfico de los radios: W06 (PA21), Chavas y ROCLOUD

Para cada posición de Cristóbal 2020 y Odile 2014:

1. Perfil radial V(r) sobre la misma malla de distancias al centro: perfil de ITCHI (`wind.py`), W06 con metodología PA21 y Chavas (perfil completo de `tcwindprofile`; si no está instalado, el modelo exterior E04 desde R34). Se marcan R34 observado, ROCLOUD (Rp y rango de cuadrantes) y los radios R(2 m/s) de W06 y Chavas.
2. Mapa del alcance de cada radio en esa posición: R34 observado y ROCLOUD por cuadrante; W06 PA21 y Chavas simétricos.
3. Mapa de la envolvente del evento: todos los contornos de la trayectoria.

Solo requiere IBTrACS y ROCLOUD (sin MSWEP).

In [ ]:
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from itchi.config import get_project_root, load_default_config

ROOT = get_project_root()
sys.path.insert(0, str(ROOT / "notebooks"))
import itchi.itchi_size_helpers as H  # noqa: E402

CFG = load_default_config()
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014"),
}
R_KM = np.linspace(1.0, 1600.0, 800)       # misma malla radial para todos los perfiles
N_COLS = 6                                  # paneles por fila
TIMES_TO_MAP = None                         # None: todas las posiciones; o lista de Timestamps
SAVE_EACH_POSITION = True                  # un PNG por posición y caso
MAP_HALF_WIDTH_DEG = 16.0

FIG_DIR = ROOT / CFG["outputs"]["figures"] / "perfiles_y_alcance"
OUT_DIR = ROOT / CFG["outputs"]["diagnostics"] / "perfiles_y_alcance"
for d in (FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

STYLE = {  # contorno/radio: (color, estilo)
    "R34 obs": ("k", "-"), "ROCLOUD": ("0.45", "-"), "W06 PA21 R(2 m/s)": ("C3", "-"),
    "Chavas R(2 m/s)": ("C2", "-"), "Chavas r0": ("C2", ":"),
}
PROFILE_COLORS = {"Rankine (wind.py)": "C0", "W06 PA21": "C3",
                  "Chavas (tcwindprofile)": "C2", "Chavas E04 (r >= R34)": "C2"}

## 1. Trayectorias y radios

In [ ]:
tracks = {}
for case, c in CASES.items():
    t = H.load_case_track(ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{c['basin']}.v04r01.nc",
                          ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{c['basin']}.csv",
                          c["name"], c["season"], c["atcf"])
    tracks[case] = H.add_chavas_radii(t, verbose_first=(case == next(iter(CASES))))
    print(case, len(t), "posiciones | fuente Chavas:",
          tracks[case]["chavas_source"].value_counts().to_dict())

cols = ["time", "lat", "lon", "vmax_kt", "rmw_km", "r34_obs_km", "pa21_rmax_km", "pa21_r34_km",
        "pa21_r2_km", "chavas_r2_km", "chavas_r0_km", "chavas_source", "rocloud_mean_km"] + H.ROC_COLS
radii_table = pd.concat([t[cols].assign(case=case) for case, t in tracks.items()])
radii_table.to_csv(OUT_DIR / "radios_por_posicion.csv", index=False)

## 2. Perfiles radiales por posición (misma distancia al centro)

In [ ]:
def plot_profile(ax, row: pd.Series) -> None:
    for label, v in H.radial_profiles(row, R_KM).items():
        ls = "--" if "E04" in label else "-"
        ax.plot(R_KM, v, color=PROFILE_COLORS[label], ls=ls, lw=1.3, label=label)
    ax.axhline(H.V_OUTER_MS, color="0.6", lw=0.7, ls=":")
    ax.axhline(34 * 0.514444, color="0.6", lw=0.7, ls=":")
    roc = np.array([row[c] for c in H.ROC_COLS], dtype=float)
    if np.isfinite(roc).any():
        ax.axvspan(np.nanmin(roc), np.nanmax(roc), color="0.85", zorder=0)
        ax.axvline(row["rocloud_mean_km"], color=STYLE["ROCLOUD"][0], lw=1.2, label="ROCLOUD Rp")
    for key, label in (("r34_obs_km", "R34 obs"), ("pa21_r2_km", "W06 PA21 R(2 m/s)"),
                       ("chavas_r2_km", "Chavas R(2 m/s)")):
        if np.isfinite(row[key]):
            color, ls = STYLE[label]
            ax.axvline(row[key], color=color, ls="--" if key != "r34_obs_km" else ls,
                       lw=1.0, label=label)
    ax.set_title(f"{row['time']:%m-%d %H}Z  {row['vmax_kt']:.0f} kt", fontsize=8)
    ax.set_xlim(0, R_KM[-1])
    ax.set_ylim(0, None)
    ax.tick_params(labelsize=7)


def plot_profiles(case: str, times=None) -> plt.Figure:
    t = tracks[case] if times is None else tracks[case][tracks[case]["time"].isin(times)]
    n_rows = int(np.ceil(len(t) / N_COLS))
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(3.2 * N_COLS, 2.4 * n_rows),
                             sharex=True, squeeze=False)
    for ax, (_, row) in zip(axes.flat, t.iterrows()):
        plot_profile(ax, row)
    for ax in axes.flat[len(t):]:
        ax.set_visible(False)
    legend = {}
    for ax in axes.flat:
        for handle, label in zip(*ax.get_legend_handles_labels()):
            legend.setdefault(label, handle)
    fig.legend(legend.values(), legend.keys(), loc="upper center", ncol=4, fontsize=8,
               frameon=False)
    fig.supxlabel("distancia al centro (km)")
    fig.supylabel("V (m s$^{-1}$)")
    fig.suptitle(case, y=1.01)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig


for case in tracks:
    fig = plot_profiles(case, TIMES_TO_MAP)
    fig.savefig(FIG_DIR / f"perfiles_{case.lower()}.png", dpi=130, bbox_inches="tight")

## 3. Alcance geográfico de cada radio por posición

In [ ]:
from dataclasses import replace

from itchi.plotting import MapStyle, configure_map_axis

PC = ccrs.PlateCarree()
EXTENT_STYLE = MapStyle(
    lon_tick_spacing=8.0, lat_tick_spacing=8.0, grid_spacing=4.0,
    grid_color="w", grid_alpha=0.6, label_size=9.0,
)


def style_for_bounds(lon_min, lon_max, lat_min, lat_max, base: MapStyle = EXTENT_STYLE):
    # Extensión alineada a múltiplos del espaciado de ticks
    step = base.lon_tick_spacing
    return replace(base,
                   lon_extent=(np.floor(lon_min / step) * step, np.ceil(lon_max / step) * step),
                   lat_extent=(np.floor(lat_min / step) * step, np.ceil(lat_max / step) * step))


def style_for_position(row: pd.Series, half_width: float = MAP_HALF_WIDTH_DEG,
                       base: MapStyle = EXTENT_STYLE) -> MapStyle:
    return style_for_bounds(row["lon"] - half_width, row["lon"] + half_width,
                            row["lat"] - 0.75 * half_width, row["lat"] + 0.75 * half_width, base)


def outline_for(label: str, row: pd.Series):
    lat, lon = float(row["lat"]), float(row["lon"])
    if label == "R34 obs":
        return H.quadrant_outline(lat, lon, H.r34_radii_km(row))
    if label == "ROCLOUD":
        return H.quadrant_outline(lat, lon, H.rocloud_radii(row))
    column = {"W06 PA21 R(2 m/s)": "pa21_r2_km", "Chavas R(2 m/s)": "chavas_r2_km",
              "Chavas r0": "chavas_r0_km"}[label]
    return H.circle_outline(lat, lon, row[column])


def draw_extent(ax, row: pd.Series, track: pd.DataFrame, legend: bool = False,
                lw: float = 1.4) -> None:
    ax.plot(track["lon"], track["lat"], color="0.3", lw=0.7, transform=PC, zorder=5)
    for label, (color, ls) in STYLE.items():
        x, y = outline_for(label, row)
        if x.size and np.isfinite(x).any():
            ax.plot(x, y, color=color, ls=ls, lw=lw, label=label,
                    transform=ccrs.Geodetic(), zorder=6)
    ax.plot(row["lon"], row["lat"], "k*", ms=9, transform=PC, zorder=7)
    if legend:
        ax.legend(fontsize=7, loc="lower left", framealpha=0.9)


def plot_extents(case: str, times=None, ncols: int = 3) -> plt.Figure:
    # Rejilla: una posición por panel
    track = tracks[case]
    t = track if times is None else track[track["time"].isin(times)]
    nrows = int(np.ceil(len(t) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows),
                             subplot_kw={"projection": PC}, squeeze=False)
    fig.subplots_adjust(wspace=0.25, hspace=0.3)
    for i, (ax, (_, row)) in enumerate(zip(axes.flat, t.iterrows())):
        configure_map_axis(ax, crs=PC, style=style_for_position(row))
        draw_extent(ax, row, track, legend=(i == 0))
        ax.set_title(f"{row['time']:%Y-%m-%d %H}Z  {row['vmax_kt']:.0f} kt", fontsize=9)
    for ax in axes.flat[len(t):]:
        ax.set_visible(False)
    fig.suptitle(f"{case}: alcance de los radios por posición", fontsize=13)
    return fig


def plot_extent_position(case: str, time: pd.Timestamp, half_width: float = MAP_HALF_WIDTH_DEG,
                         figsize=(9, 7), style: MapStyle | None = None):
    # Un mapa a detalle para una posición; None si la posición no existe
    track = tracks[case]
    rows = track[track["time"] == time]
    if rows.empty:
        print(f"{case}: sin posición en {time}; se omite")
        return None
    row = rows.iloc[0]
    base = style or replace(EXTENT_STYLE, label_size=12.0, tick_size=7.0)
    fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": PC})
    configure_map_axis(ax, crs=PC, style=style_for_position(row, half_width, base))
    draw_extent(ax, row, track, legend=True, lw=1.8)
    ax.set_title(f"{case} {row['time']:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt | "
                 f"Chavas: {row['chavas_source']}", fontsize=12)
    return fig


for case in tracks:
    fig = plot_extents(case, TIMES_TO_MAP)
    fig.savefig(FIG_DIR / f"alcance_por_posicion_{case.lower()}.png", dpi=120,
                bbox_inches="tight")

In [ ]:
def plot_profile_and_extent(case: str, row: pd.Series, half_width: float = MAP_HALF_WIDTH_DEG):
    # Perfil radial (izquierda) y alcance geográfico (derecha) de una posición
    track = tracks[case]
    fig = plt.figure(figsize=(13, 5))
    ax_p = fig.add_subplot(1, 2, 1)
    plot_profile(ax_p, row)
    ax_p.legend(fontsize=7, loc="upper right")
    ax_p.set(xlabel="distancia al centro (km)", ylabel="V (m s$^{-1}$)")

    ax_m = fig.add_subplot(1, 2, 2, projection=PC)
    configure_map_axis(ax_m, crs=PC, style=style_for_position(row, half_width))
    draw_extent(ax_m, row, track, legend=True)

    fig.suptitle(f"{case} {row['time']:%Y-%m-%d %H} UTC | Vmax {row['vmax_kt']:.0f} kt | "
                 f"Chavas: {row['chavas_source']}", fontsize=12)
    fig.subplots_adjust(wspace=0.25)
    return fig


# Opcional: un PNG por posición (perfil + mapa lado a lado)
if SAVE_EACH_POSITION:
    for case, t in tracks.items():
        out = FIG_DIR / case.lower()
        out.mkdir(exist_ok=True)
        for _, row in t.iterrows():
            fig = plot_profile_and_extent(case, row)
            fig.savefig(out / f"{row['time']:%Y%m%d%H}.png", dpi=120, bbox_inches="tight")
            plt.close(fig)

## 4. Envolvente del evento: contornos de todas las posiciones

In [ ]:
fig, axes = plt.subplots(1, len(tracks), figsize=(8 * len(tracks), 6.5),
                         subplot_kw={"projection": PC}, squeeze=False)
fig.subplots_adjust(wspace=0.2)
for ax, (case, t) in zip(axes[0], tracks.items()):
    configure_map_axis(ax, crs=PC, style=style_for_bounds(
        t["lon"].min() - 12, t["lon"].max() + 12, t["lat"].min() - 10, t["lat"].max() + 10))
    ax.plot(t["lon"], t["lat"], "k.-", lw=1.0, ms=3, transform=PC, zorder=7)
    for label in ("ROCLOUD", "W06 PA21 R(2 m/s)", "Chavas R(2 m/s)", "R34 obs"):
        color, ls = STYLE[label]
        labeled = False
        for _, row in t.iterrows():
            x, y = outline_for(label, row)
            if x.size and np.isfinite(x).any():
                ax.plot(x, y, color=color, ls=ls, lw=0.6, alpha=0.6,
                        label=None if labeled else label, transform=ccrs.Geodetic(), zorder=6)
                labeled = True
    ax.legend(fontsize=8, loc="lower left", framealpha=0.9)
    ax.set_title(f"{case}: alcance de los radios en toda la trayectoria", fontsize=11)
fig.savefig(FIG_DIR / "envolvente_eventos.png", dpi=130, bbox_inches="tight")